In [1]:
# 6-26-2026

In [2]:
import xarray as xr
import numpy as np
import pandas as pd

In [3]:
ds_path = "../data/seasfire_pyromes_ecoregions.zarr"
domains_path = "../data/ecoregion_domains.zarr"

In [ ]:
# this section is for getting fire sparsity
# its calculated as (num fire cells)/(total cells) per domain

In [4]:
ds = xr.open_zarr(ds_path, consolidated=True)
domains = xr.open_zarr(domains_path, consolidated=True)

In [8]:
domain_arr = domains["domain_id"].values
ba = ds["gwis_ba"].values  # (time, lat, lon)

In [9]:
unique_domains = np.unique(domain_arr)
unique_domains = unique_domains[unique_domains != -1]

In [10]:
sparsity = {}
for domain in unique_domains:
    mask = (domain_arr == domain)
    domain_ba = ba[:, mask].ravel()  # all (lat, lon, time) entries for this domain
    total = len(domain_ba)
    fire = (domain_ba > 0).sum()
    sparsity[int(domain)] = fire / total

In [11]:
sparsity_df = pd.DataFrame.from_dict(sparsity, orient="index", columns=["fire_sparsity"])
sparsity_df.index.name = "domain_id"

In [ ]:
sparsity_df.head(15) # good, domains like 11 have high value

,fire_sparsity
domain_id,
0,0.081824
1,0.013560
2,0.032987
3,0.001419
4,0.007227
5,0.058533
6,0.003038
7,0.004830
8,0.019789


In [14]:
# this section is for land cover diversity, will use simpson diversity index formula

In [17]:
lccs_vars = ["lccs_class_1", "lccs_class_2", "lccs_class_3", "lccs_class_4", "lccs_class_6", "lccs_class_7"]

# build a (num_classes, num_domains) array of mean proportions
domain_props = np.zeros((len(lccs_vars), len(unique_domains)))

In [18]:
for i, var in enumerate(lccs_vars):
    print(f"loading {var}...")
    data = ds[var].values  # (time, lat, lon)
    for j, domain in enumerate(unique_domains):
        mask = (domain_arr == domain)
        domain_props[i, j] = np.nanmean(data[:, mask])
    del data

diversity = {}
for j, domain in enumerate(unique_domains):
    p = domain_props[:, j]
    p = p[p > 0]
    p = p / p.sum() # normalize jus in case
    diversity[int(domain)] = 1 - np.sum(p ** 2)

loading lccs_class_1...
loading lccs_class_2...
loading lccs_class_3...
loading lccs_class_4...
loading lccs_class_6...
loading lccs_class_7...


In [19]:
diversity_df = pd.DataFrame.from_dict(diversity, orient="index", columns=["land_cover_diversity"])
diversity_df.index.name = "domain_id"

In [21]:
diversity_df.head(10)

,land_cover_diversity
domain_id,
0,0.728328
1,0.218936
2,0.602900
3,0.365885
4,0.615066
5,0.560294
6,0.626834
7,0.655366
8,0.553258


In [22]:
# will do fire seasonality later